# 06 — Class imbalance comparison

Train the **same** text-only pipeline twice on **the same** stratified training rows. The only difference is `class_weight="balanced"`, which computes inverse-frequency weights from the training labels. Validation/test class distributions stay untouched. Use the same positive-class mapping as notebook 05; no resampling or dataset alterations are performed.


In [ ]:
from pathlib import Path
import sys

# Run Jupyter from the repository root or the notebooks directory.
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if not (ROOT / "src").exists():
    raise RuntimeError("Start Jupyter in the CrossHealth-Risk repository")
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.models.baseline import build_baseline, split_labeled_data
from src.evaluation.metrics import evaluate_binary_classifier
from src.evaluation.plots import plot_confusion, plot_roc_and_pr
from src.models.balanced_model import build_balanced_baseline
from src.evaluation.plots import plot_metric_comparison


## Set the same processed file, columns and positive label as notebook 05


In [ ]:
# Fill these using Member 1's actual standardized output. No columns are assumed.
DATA_PATH = ROOT / "data" / "processed" / "SET_MEMBER1_FILENAME.csv"
TEXT_COLUMN = None  # example: the actual standardized text column name
LABEL_COLUMN = None  # example: the actual binary ground-truth column name
POSITIVE_LABEL = None  # exact observed label for the misinformation/risk class
SEED = 42

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Set DATA_PATH to Member 1's processed CSV: {DATA_PATH}")
if not all(value is not None for value in (TEXT_COLUMN, LABEL_COLUMN, POSITIVE_LABEL)):
    raise ValueError("Set TEXT_COLUMN, LABEL_COLUMN and POSITIVE_LABEL from Member 1's actual data")

data = pd.read_csv(DATA_PATH)
if LABEL_COLUMN not in data:
    raise KeyError(f"Missing label column {LABEL_COLUMN!r}; available: {list(data.columns)}")
print("Rows:", len(data), "| observed label counts:", data[LABEL_COLUMN].value_counts(dropna=False).to_dict())
if POSITIVE_LABEL not in set(data[LABEL_COLUMN].dropna()):
    raise ValueError("POSITIVE_LABEL must exactly match one observed label, including its type")
split = split_labeled_data(
    data, text_column=TEXT_COLUMN, label_column=LABEL_COLUMN, random_state=SEED
)
print("Train / validation / test:", len(split.train), len(split.validation), len(split.test))


## Fit on the training split only


In [ ]:
models = {
    "Unweighted": build_baseline(random_state=SEED),
    "Class weighted": build_balanced_baseline(random_state=SEED),
}
for model in models.values():
    model.fit(split.train[TEXT_COLUMN], split.train[LABEL_COLUMN])
print("Training label counts:", split.train[LABEL_COLUMN].value_counts().to_dict())


## Inspect validation before test

Fixed model settings and a fixed 0.5 decision threshold keep this comparison controlled. If you later tune settings or thresholds, use training/validation only and disclose the selection procedure before evaluating the untouched test set.


In [ ]:
validation_results = [
    evaluate_binary_classifier(
        model, split.validation[TEXT_COLUMN], split.validation[LABEL_COLUMN],
        positive_label=POSITIVE_LABEL, name=f"{name} validation",
    )
    for name, model in models.items()
]
display(pd.DataFrame({result.name: result.metrics for result in validation_results}).T)


## One held-out test comparison


In [ ]:
test_results = [
    evaluate_binary_classifier(
        model, split.test[TEXT_COLUMN], split.test[LABEL_COLUMN],
        positive_label=POSITIVE_LABEL, name=name,
    )
    for name, model in models.items()
]
display(pd.DataFrame({result.name: result.metrics for result in test_results}).T)
for result in test_results:
    print(result.name, "[[TN, FP], [FN, TP]]:\n", result.confusion)
    display(plot_confusion(result))
    plt.close("all")
display(plot_metric_comparison(test_results))
display(plot_roc_and_pr(test_results))
plt.close("all")


## Paper reporting notes

Report class prevalence, confusion matrices, positive-class precision/recall/F1, ROC AUC and average precision with the exact data version and split seed. A single split does not establish statistical significance. If documents from the same source or underlying claim can appear in multiple splits, agree on a group-aware split with Member 1 *before* using these results in a paper. Do not claim the balanced model wins unless the observed results support it.
